In [ ]:
# Versione 6 giugno: pipeline completa con tanto di Leave-One-Run-Out Cross Validation

import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
import numpy as np
from sklearn import preprocessing
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from pathlib import Path
from util.preprocessing import create_sliding_windows, create_window_labels
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)
mne.set_log_level('WARNING')

root = "../data"
root = (Path(root).resolve())
runs = ["5", "9", "13"]   # Le run che vengono prese in considerazione
test_run_order = [runs[2], runs[1], runs[0]]  # Ordine in cui vengono usate le run come test set
first_person = 1
people = 10
all_accuracy = np.zeros((people, len(test_run_order)))  # Accuratezza per soggetto e per run di test

# Eseguiamo la scansione di tutti i soggetti 
for i in range(first_person, first_person + people):
    subject = f"{i:03d}"

    for test_index, test_run in enumerate(test_run_order):
        train_runs = [run for run in runs if run != test_run]
        X_train = []
        y_train = []

        X_test = []
        y_test = []

        # Per ogni soggetto eseguiamo la scansione sulle run di nostro interesse
        for run in runs:
            bids_path = BIDSPath(   # Specifichiamo il percorso del dataset e BIDS eseguirà correttamente la scansione 
                subject=subject,
                task="motion",
                run=run,
                datatype="eeg",
                root=root,
            )
            
            try:
                # Fase 1: lettura dei dati
                raw = read_raw_bids(bids_path, verbose=False)  
                events, event_id = mne.events_from_annotations(raw, verbose=False) 
                raw.load_data(verbose=False) # Carico i dati in memoria per poter filtrare ecc.
                raw.filter(l_freq=8, h_freq=30, verbose=False)  # Filtro passa banda 1-30 Hz
                raw.set_eeg_reference('average', projection=False, verbose=False)  # Riferimento medio
                
                # Applico ICA su tutti i canali
                ica = mne.preprocessing.ICA(n_components=15, method='fastica', random_state=42, verbose=False)
                ica.fit(raw, verbose=False)
                eog_indices, _ = ica.find_bads_eog(raw, ch_name=["Fp1", "Fp2"], verbose=False)
                muscle_indices, _ = ica.find_bads_muscle(raw, verbose=False)
                ica.exclude = list(set(eog_indices + muscle_indices))
                ica.apply(raw, verbose=False)
                # print(f"Soggetto {subject} run {run} - Componenti rimosse: {len(ica.exclude)} {ica.exclude}")
                
                event_map = {
                    event_id['TASK3T0']: 1,
                    event_id['TASK3T1']: 2,
                    event_id['TASK3T2']: 3
                }
                sfreq = raw.info['sfreq']  # frequenza di campionamento

                # Fase 2: pre-processing
                window_size = 2  # Lunghezza finestra in secondi
                step_size = 0.5  # Lunghezza passo in secondi

                windows, window_samples, step_samples, total_samples = create_sliding_windows(
                    raw,
                    window_size,
                    step_size
                )        

                C_channels = ["C1", "C2", "C3", "C4", "C5", "C6", "Cz"]
                CP_channels = [ch for ch in raw.ch_names if ch.startswith("Cp")]
                FC_channels = [ch for ch in raw.ch_names if ch.startswith("Fc")]
                T_channels =  [ch for ch in raw.ch_names if ch.startswith("T")]
                F_channels =  ["F1", "F2", "F3", "F4", "F5", "F6"]
                P_channels =  ["P1", "P2", "P3", "P4", "P5", "P6"]
                channels_of_interest = C_channels  + FC_channels + CP_channels 
                channels_of_interest = ["C3", "C4", "Cz", "Fc3", "Fc4", "Fcz", "Cp3", "Cp4", "Cpz"]
                picks = mne.pick_channels(raw.ch_names, channels_of_interest)
                windows = windows[:, picks, :]

                y = create_window_labels(
                    events,
                    event_map,
                    total_samples,
                    window_samples,
                    step_samples,
                    threshold= 0.8
                )
                
                # Divido la classificazione in due step: prima distinguo tra stato di riposo e di attivazione.
                # In caso di attivazione, distinguo tra sinistra e destra. Per ora faccio solo la prima parte.
                y_rest_active = np.where(y == 1, 0, 1)  
                 
                # Variabile per addestrare il modello sul sinistra/destra (da commentare se non si vuole usare)
                mask_active = y != 1
                y_active = y[mask_active]
                y_lr = np.where(y_active == 2, 0, 1)

                # Assegno a y_binary le etichette desiderate (se voglio rest/active commento la seconda riga)
                y_binary = y_lr
                windows = windows[mask_active]
                

                if run in train_runs:
                    X_train.append(windows)
                    y_train.append(y_binary)
                else:
                    X_test.append(windows)
                    y_test.append(y_binary)

                # print(f"Soggetto {subject} run {run} - Campioni: {X_csp.shape[0]}, Feature per campione: {X_csp.shape[1]}, Etichette: {y.shape[0]}")

            except Exception as e:
                print(f"Errore {subject}: {e}")

        X_train = np.concatenate(X_train, axis=0)
        y_train = np.concatenate(y_train)

        X_test = np.concatenate(X_test, axis=0)
        y_test = np.concatenate(y_test)

        pipe = Pipeline([
            ("csp", CSP(reg='ledoit_wolf')), # reg aiuta la stabilità con finestre corte
            ("scaler", StandardScaler()),    # Fondamentale per SVM
            ("svm", SVC(probability=True))
        ])

        param_grid = {
            "csp__n_components": [2, 4, 6],
            "csp__log": [True],
            "svm__C": [0.1, 1, 10, 100],
            "svm__gamma": ["scale"], 
            "svm__kernel": ["rbf", "linear"]
        }

        grid = GridSearchCV(
            pipe,
            param_grid,
            cv=5,
            scoring="balanced_accuracy",
            n_jobs=-1
        )

        grid.fit(X_train, y_train)
        
        # Introduzione di logica probabilistica
        probs = grid.predict_proba(X_test)
        max_probs = np.max(probs, axis=1)
        predictions = np.argmax(probs, axis=1)
        print("----------------------------------------------------")
        print(f"Paziente {subject} - Run di test: {test_run}")
        # print("Min:", np.min(max_probs))
        # print("Mean:", np.mean(max_probs))
        # print("Max:", np.max(max_probs))
        # print("Best score:", grid.best_score_)
        print(
            balanced_accuracy_score(
                y_test,
                predictions
            )
        )
        threshold = 0.64
        accepted_mask = max_probs >= threshold

        accepted_predictions = predictions[accepted_mask]
        accepted_true_labels = y_test[accepted_mask]

        # Balanced misura un'accuratezza media tra le classi (così se ho 70% su una classe e 0 sull'altra ottengo comunque un valore adeguato)
        accuracy = balanced_accuracy_score(
            accepted_true_labels,
            accepted_predictions
        )
        
        # # Coonfusion matrix
        # cm = confusion_matrix(
        #     accepted_true_labels,
        #     accepted_predictions
        # )
        # print(cm)
        # cm = confusion_matrix(
        #     y_test,
        #     predictions
        # )
        # print(cm)

        all_accuracy[i-first_person, test_index] = accuracy
        accepted = np.sum(accepted_mask)
        total = len(y_test)
        discarded = total - accepted
        print(f"Campioni totali/accettati/scartati: {total}/{accepted}/{discarded}")
        print(f"Percentuale scartata: {100*discarded/total:.2f}%")
        print(f"Balanced accuracy sui campioni accettati: {accuracy*100:.2f}%")
        

    patient_mean = all_accuracy[i-first_person].mean()
    print(f"Accuratezza media paziente {subject}: {patient_mean}")

mean_accuracy = all_accuracy.mean()
print()
print(f"Accuratezza media: {mean_accuracy}")

std_accuracy = all_accuracy.std()
print(f"Deviazione standard: {std_accuracy}")

----------------------------------------------------
Paziente 021 - Run di test: 13
0.8095238095238095
Campioni totali/accettati/scartati: 84/68/16
Percentuale scartata: 19.05%
Balanced accuracy sui campioni accettati: 86.62%
----------------------------------------------------
Paziente 021 - Run di test: 9
0.8809523809523809
Campioni totali/accettati/scartati: 84/78/6
Percentuale scartata: 7.14%
Balanced accuracy sui campioni accettati: 89.93%
----------------------------------------------------
Paziente 021 - Run di test: 5
0.7619047619047619
Campioni totali/accettati/scartati: 84/75/9
Percentuale scartata: 10.71%
Balanced accuracy sui campioni accettati: 82.03%
Accuratezza media paziente 021: 0.8619740639477481

Accuratezza media: 0.8619740639477481
Deviazione standard: 0.032390248597697235
